# Senate Profile LLM Extraction Pipeline — v4
**DSBA 6010 — Chloe Partridge**

Aligned with Liu et al. (USENIX Security 2025) *Evaluating LLM-based Personal Information Extraction and Countermeasures*.

Key additions over v3:
- **Prompt-style ablation** (direct / pseudocode) per Section 4.2 / Table 13
- **In-context learning examples** for Task 1 per Section 6.2 / Figure 2
- **Traditional baselines** (regex + spaCy NER) per Tables 4–5
- **Evaluation metrics** (Accuracy, Rouge-1, BERT score) per Section 6.1.4
- **Model comparison** (8B vs 70B) per Table 3 / Section 6.2

Two tasks remain **separate API calls** to prevent party-label leakage into ideological inference:
- **Task 2 runs first** on party-sanitized text
- **Task 1 runs second** on full text

**Model:** Groq (free tier) — config loaded from `groq_config_extraction.json`

## 1. Dependencies

In [1]:
# !pip install beautifulsoup4 groq pandas tqdm rouge-score bert-score spacy
# !python -m spacy download en_core_web_sm

  Using cached rouge_score-0.1.2-py3-none-any.whl
     |████████████████████████████████| 1.3 MB 6.4 MB/s eta 0:00:01
  Installing build dependencies ... error
  ERROR: Command errored out with exit status 1:
   command: /Users/chloe/LLM-Based-Personal-Profile-Extraction/pie_env/bin/python3 /private/var/folders/xz/3l5pw91d3_388t5z0w3_5dmw0000gn/T/pip-standalone-pip-jhboxsio/__env_pip__.zip/pip install --ignore-installed --no-user --prefix /private/var/folders/xz/3l5pw91d3_388t5z0w3_5dmw0000gn/T/pip-build-env-3f29_teb/overlay --no-warn-script-location --no-binary :none: --only-binary :none: -i https://pypi.org/simple -- setuptools 'cython>=3.0,<4.0' 'cymem>=2.0.2,<2.1.0' 'preshed>=3.0.2,<3.1.0' 'murmurhash>=0.28.0,<1.1.0' 'thinc>=8.3.12,<8.4.0' 'numpy>=2.0.0,<3.0.0'
       cwd: None
  Complete output (14 lines):
    Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
    Using cached cython-3.2.4-cp39-cp39-macosx_11_0_arm64.whl (3.0 MB)
    Using cached cymem-2.0.13-cp39-cp39-macos

In [2]:
import json, time, re
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from groq import Groq
from rouge_score import rouge_scorer
import bert_score
import spacy

# Load spaCy model for baseline NER
nlp = spacy.load('en_core_web_sm')

/Users/chloe/LLM-Based-Personal-Profile-Extraction/pie_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Configuration

Loads from `model_config_extraction.json`.

> **Before running:** if you have a `results_raw.json` from a previous run, rename it:
> ```bash
> mv senate_results/results_raw.json senate_results/results_raw_v1_backup.json
> ```


In [3]:
HTML_DIR   = Path("./senate_html")
OUTPUT_DIR = Path("./senate_results")
OUTPUT_DIR.mkdir(exist_ok=True)

with open("configs/model_configs/groq_config_extraction.json") as f:
    config = json.load(f)

api_key = config["api_key_info"]["api_keys"][0]
model   = config["model_info"]["name"]
temp    = config["params"]["temperature"]
max_tok = config["params"]["max_output_tokens"]

client = Groq(api_key=api_key)

html_files = sorted(HTML_DIR.glob("*.html"))
print(f"Model:       {model}")
print(f"Temperature: {temp}  |  Max tokens: {max_tok}")
print(f"HTML files:  {len(html_files)}")


Model:       llama-3.1-8b-instant
Temperature: 0.1  |  Max tokens: 1500
HTML files:  101


## 3. HTML Preprocessing

Strips `<script>`, `<style>`, `<nav>`, `<footer>` — noise with no informational value.  
Content preserved exactly as written.


In [4]:
PARTY_RE = re.compile(
    r"\b(Republican|Democrat|Democratic|GOP|Independent)\b", re.IGNORECASE
)

def extract_readable_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s{2,}", " ", text).strip()

sample = html_files[0]
text   = extract_readable_text(sample.read_text(encoding="utf-8", errors="ignore"))
print(f"File:         {sample.name}")
print(f"Raw size:     {sample.stat().st_size:,} chars")
print(f"Cleaned size: {len(text):,} chars")
print(f"\n--- First 500 chars ---\n{text[:500]}")


File:         Adam_B_Schiff_CA.html
Raw size:     222,779 chars
Cleaned size: 7,918 chars

--- First 500 chars ---
About - Senator Schiff Skip to content Home Search Mobile Nav Open Open Search Close Search Contact Adam Close Mobile Nav Search About Senator Adam Schiff About Senator Adam Schiff After Adam graduated from law school, he moved to Los Angeles to serve as a law clerk for Judge William Matthew Byrne, Jr. Adam later joined the U.S. Attorney’s Office in Los Angeles as a federal prosecutor, where he served for almost six years, most notably prosecuting, Richard Miller, the first FBI agent ever to be 


## 4. Prompt Design

### Why separate calls?
In a combined prompt, the model resolves `party` during Task 1 and carries that knowledge
into Task 2 within the same reasoning pass — inflating Task 2 confidence scores and
undermining the core claim that ideology is inferable from language alone.

Running them separately with Task 2 first ensures the model has no resolved party label
when making its ideological inference.

### Prompt style ablation (Liu et al. Section 4.2, Table 13)
Liu et al. test four styles: direct, persona, contextual, and pseudocode.
Pseudocode outperforms direct for structured fields (email, phone, name).
We implement both **direct** and **pseudocode** for Task 1 to replicate their finding.

### In-context learning (Liu et al. Section 6.2, Figure 2)
The paper shows ICL has the largest impact on occupation-type fields.
We add **one demonstration example** to Task 1 to test this on `committee_roles` and `party`.

### Task 2 — Ideological Inference (runs first)
Input: party-sanitized text (all party names replaced with `[PARTY]`).  
The model reasons purely from values language, policy framing, and identity signals.

### Task 1 — Structured PII Extraction (runs second)
Input: full cleaned text.  
Extracts discrete fields including explicit party label.

In [8]:
# ── Task 2: Ideology inference (unchanged from v3) ──────────────────────────
TASK2_PROMPT = """You are a political communication analyst.
Infer the ideological leaning of a U.S. Senator based ONLY on the language,
framing, values, and identity signals in their profile — NOT from any party label.
All party names have been replaced with [PARTY] — ignore them entirely.

Return ONLY valid JSON. No preamble, no markdown fences.

{
  "ideological_label": one of ["Liberal", "Conservative", "Moderate", "Unclear"],
  "confidence": float 0.0-1.0,
  "summary": "2-3 sentences citing specific language from the profile"
}
"""

# ── Task 1 — DIRECT style (Liu et al. "direct" prompt, Table 13) ──────────
TASK1_DIRECT = """You are a precise data extraction assistant.
Extract the following fields from the senator profile text.
Return ONLY valid JSON. No preamble, no markdown fences.

{
  "full_name": string or null,
  "birthdate": "YYYY-MM-DD" or null,
  "birth_year_inferred": integer or null,
  "education": [{"degree": string, "institution": string, "year": integer or null}],
  "party": string or null,
  "committee_roles": [string],
  "gender": string or null,
  "race_ethnicity": string or null
}

Rules:
- birth_year_inferred: only if birthdate is null AND age is mentioned
- race_ethnicity: ONLY if explicitly stated; otherwise null
- Never guess; return null for missing fields
"""

# ── Task 1 — PSEUDOCODE style (Liu et al. Section 4.2, best for structured fields) ──
# The paper finds pseudocode achieves slightly better results for name, email,
# mailing address, and phone (Table 13). We apply it here for the analogous
# structured fields in the senator schema.
TASK1_PSEUDOCODE = """// extract_senator_profile(profile: str) -> dict
// Extracts structured personal information from a U.S. Senator's public profile.
// Return ONLY valid JSON matching the schema below. No preamble, no markdown fences.
//
// Schema:
// {
//   "full_name": string or null,
//   "birthdate": "YYYY-MM-DD" or null,
//   "birth_year_inferred": integer or null,
//   "education": [{"degree": string, "institution": string, "year": int or null}],
//   "party": string or null,
//   "committee_roles": [string],
//   "gender": string or null,
//   "race_ethnicity": string or null
// }
// Rules:
// - birth_year_inferred: only if birthdate is null AND age is mentioned
// - race_ethnicity: ONLY if explicitly stated; otherwise null
// - Never guess; return null for missing fields
//
// PROFILE TEXT:
"""

# ── Task 1 — DIRECT + IN-CONTEXT LEARNING (Liu et al. Section 6.2, Figure 2) ──
# The paper shows ICL has the largest impact on occupation-type fields.
# One demonstration example is provided for party and committee_roles,
# which are the closest analogs to "occupation" and "affiliation" in our schema.
TASK1_ICL = """You are a precise data extraction assistant.
Extract the following fields from the senator profile text.
Return ONLY valid JSON. No preamble, no markdown fences.

EXAMPLE INPUT:
Senator Jane Smith is a Democrat from Ohio. She serves on the Senate Finance
Committee and the Senate Judiciary Committee. She earned a J.D. from Harvard
Law School in 1995 and a B.A. from Ohio State University in 1992.

EXAMPLE OUTPUT:
{"full_name": "Jane Smith", "birthdate": null, "birth_year_inferred": null,
 "education": [{"degree": "J.D.", "institution": "Harvard Law School", "year": 1995},
               {"degree": "B.A.", "institution": "Ohio State University", "year": 1992}],
 "party": "Democrat", "committee_roles": ["Senate Finance Committee",
 "Senate Judiciary Committee"], "gender": null, "race_ethnicity": null}

NOW EXTRACT:
{
  "full_name": string or null,
  "birthdate": "YYYY-MM-DD" or null,
  "birth_year_inferred": integer or null,
  "education": [{"degree": string, "institution": string, "year": integer or null}],
  "party": string or null,
  "committee_roles": [string],
  "gender": string or null,
  "race_ethnicity": string or null
}

Rules:
- birth_year_inferred: only if birthdate is null AND age is mentioned
- race_ethnicity: ONLY if explicitly stated; otherwise null
- Never guess; return null for missing fields
"""

# Default prompt style — change to TASK1_PSEUDOCODE or TASK1_ICL to ablate
TASK1_PROMPT = TASK1_DIRECT
PROMPT_STYLE  = "direct"   # track which style produced these results

def call_groq(prompt: str, text: str, retries: int = 5,
              model_override: str = None) -> dict:
    """Single LLM call with exponential backoff. model_override for comparison runs."""
    m = model_override or model
    # Pseudocode prompt embeds the text differently
    if prompt is TASK1_PSEUDOCODE:
        full_prompt = prompt.replace('PROFILE TEXT:\n"""', f'PROFILE TEXT:\n{text[:12000]}"""')
    else:
        full_prompt = prompt + "\n\nPROFILE TEXT:\n" + text[:12000]
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=m,
                temperature=temp,
                max_tokens=max_tok,
                messages=[{"role": "user", "content": full_prompt}]
            )
            raw = response.choices[0].message.content.strip()
            raw = raw.strip("```json").strip("```").strip()
            return json.loads(raw)
        except json.JSONDecodeError as e:
            return {"error": f"JSON parse error: {e}", "raw_response": raw}
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = (2 ** attempt) * 10
                print(f"\n  Rate limited. Waiting {wait}s before retry {attempt+1}/{retries}...")
                time.sleep(wait)
            else:
                return {"error": str(e)}

def run_pipeline(text: str, model_override: str = None) -> dict:
    sanitized = PARTY_RE.sub("[PARTY]", text)

    # Task 2 first — no party context
    task2 = call_groq(TASK2_PROMPT, sanitized, model_override=model_override)
    time.sleep(1.5)

    # Task 1 second — full text, current prompt style
    task1 = call_groq(TASK1_PROMPT, text, model_override=model_override)

    return {
        "task1_pii": task1,
        "task2_ideology": task2,
        "prompt_style": PROMPT_STYLE
    }

## 5a. Traditional Baselines (Liu et al. Tables 4–5)

Liu et al. benchmark LLMs against regex, keyword search, and spaCy NER,
finding LLMs outperform traditional methods in almost all scenarios.
We implement the two most relevant baselines for our senator schema:
- **Regex** — structured fields (name, party via keyword, education year)
- **spaCy NER** — name (PERSON), affiliation/committee (ORG)

Running these on the same profiles lets us reproduce the LLM-vs-traditional comparison
in our own dataset as described in their Section 6.2.

In [ ]:
# ── Regex baseline (Liu et al. Section 6.1.3) ────────────────────────────
# Mirrors the paper's regular expression approach for fields with clear structure.
import re as _re

EMAIL_RE = _re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}")
PHONE_RE = _re.compile(r"(?:\+\d+\s?)?(?:\d{3}[\-\s]?\d{3,}[\-\s]?\d{4})")
YEAR_RE  = _re.compile(r"\b(19[4-9]\d|20[0-2]\d)\b")
NAME_RE  = _re.compile(r"\bSenator\s+([A-Z][a-z]+(?:\s+[A-Z]\.?)?\s+[A-Z][a-z]+)")
PARTY_KEYWORD_RE = _re.compile(
    r"\b(Republican|Democrat|Democratic|Independent)\b", _re.IGNORECASE
)

def regex_extract(text: str) -> dict:
    """Regex baseline — structured fields only."""
    name_m  = NAME_RE.search(text)
    party_m = PARTY_KEYWORD_RE.search(text)
    years   = YEAR_RE.findall(text)
    return {
        "full_name":  name_m.group(1) if name_m else None,
        "party":      party_m.group(0).title() if party_m else None,
        "email":      EMAIL_RE.findall(text) or None,
        "phone":      PHONE_RE.findall(text) or None,
        "years_found": years[:5] if years else None,
    }

# ── spaCy NER baseline (Liu et al. Section 6.1.3) ────────────────────────
def spacy_extract(text: str) -> dict:
    """spaCy NER baseline — PERSON, ORG entities."""
    doc = nlp(text[:10000])  # spaCy token limit safety
    persons = list({ent.text for ent in doc.ents if ent.label_ == "PERSON"})
    orgs    = list({ent.text for ent in doc.ents if ent.label_ == "ORG"})
    return {
        "persons_detected": persons[:5],
        "orgs_detected":    orgs[:10],
    }

# ── Quick smoke test on first profile ────────────────────────────────────
sample_text = extract_readable_text(
    html_files[0].read_text(encoding="utf-8", errors="ignore")
)
print("=== Regex baseline ===")
print(json.dumps(regex_extract(sample_text), indent=2))
print("\n=== spaCy NER baseline ===")
print(json.dumps(spacy_extract(sample_text), indent=2))


## 5b. Model Comparison (Liu et al. Table 3)

Liu et al. find that larger models are more successful at PIE across all categories.
We test this on a **random sample of 10 senators** comparing:
- `llama-3.1-8b-instant` (default, free tier)
- `llama-3.3-70b-versatile` (larger, also available on Groq free tier)

We report Task 1 field coverage for each model to replicate their scaling finding.
Run this cell after the main pipeline to avoid rate limit conflicts.

In [ ]:
# ── Model comparison on a 10-senator sample (Liu et al. Table 3) ─────────
# Change COMPARISON_MODEL to any Groq model available on your account.
COMPARISON_MODEL = "llama-3.3-70b-versatile"
SAMPLE_N         = 10

import random
random.seed(42)
sample_files = random.sample(html_files, min(SAMPLE_N, len(html_files)))

comparison_results = []
for model_name in [model, COMPARISON_MODEL]:
    print(f"\nRunning {model_name} on {SAMPLE_N} profiles...")
    for hf in tqdm(sample_files, desc=model_name):
        text = extract_readable_text(hf.read_text(encoding="utf-8", errors="ignore"))
        t1   = call_groq(TASK1_PROMPT, text, model_override=model_name)
        comparison_results.append({
            "senator_id": hf.stem,
            "model":      model_name,
            "result":     t1
        })
        time.sleep(3)

# ── Summarise field coverage by model ────────────────────────────────────
T1_FIELDS = ["full_name","birthdate","birth_year_inferred","education",
             "party","committee_roles","gender","race_ethnicity"]

rows = []
for r in comparison_results:
    res = r["result"]
    row = {"senator_id": r["senator_id"], "model": r["model"]}
    for f in T1_FIELDS:
        v = res.get(f)
        row[f] = 1 if (v is not None and v != [] and v != "null") else 0
    rows.append(row)

df_cmp = pd.DataFrame(rows)
print("\n=== Field Coverage by Model (Liu et al. Table 3 analog) ===")
print(df_cmp.groupby("model")[T1_FIELDS].mean().round(2).T.to_string())
df_cmp.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)


## 5. Run Pipeline

Two API calls per senator (Task 2 → Task 1).  
Results saved incrementally — safe to interrupt and resume.


In [24]:
raw_path = OUTPUT_DIR / "results_raw.json"

if raw_path.exists():
    with open(raw_path) as f:
        results = json.load(f)
    done_ids = {r["senator_id"] for r in results}
    print(f"Resuming — {len(done_ids)} senators already processed.")
else:
    results, done_ids = [], set()

remaining = [f for f in html_files if f.stem not in done_ids]
print(f"Senators remaining: {len(remaining)}")

for html_file in tqdm(remaining, desc="Processing senators"):
    senator_id = html_file.stem
    html       = html_file.read_text(encoding="utf-8", errors="ignore")
    text       = extract_readable_text(html)
    result     = run_pipeline(text)

    results.append({"senator_id": senator_id, **result})
    with open(raw_path, "w") as f:
        json.dump(results, f, indent=2)

    time.sleep(4)

print(f"\nDone. {len(results)} senators processed.")


Senators remaining: 101


Processing senators:   0%|          | 0/101 [00:00<?, ?it/s]


Done. 101 senators processed.


## 6. Flatten Results to CSV

- `task1_pii.csv` — structured PII fields  
- `task2_ideology.csv` — ideology inference results  

Safe to run mid-pipeline to inspect partial results.


In [27]:
with open(raw_path) as f:
    results = json.load(f)

T1_FIELDS = ["full_name","birthdate","birth_year_inferred",
             "education","party","committee_roles","gender","race_ethnicity"]
T2_FIELDS = ["ideological_label","confidence","summary"]

task1_rows, task2_rows = [], []

for r in results:
    sid = r["senator_id"]

    t1  = {"senator_id": sid}
    res = r.get("task1_pii", {})
    for f in T1_FIELDS:
        val = res.get(f)
        t1[f] = json.dumps(val) if isinstance(val, list) else val
    t1["error"] = res.get("error")
    task1_rows.append(t1)

    t2  = {"senator_id": sid}
    res = r.get("task2_ideology", {})
    for f in T2_FIELDS:
        t2[f] = res.get(f)
    t2["error"] = res.get("error")
    task2_rows.append(t2)

df_t1 = pd.DataFrame(task1_rows)
df_t2 = pd.DataFrame(task2_rows)

df_t1.to_csv(OUTPUT_DIR / "task1_pii.csv", index=False)
df_t2.to_csv(OUTPUT_DIR / "task2_ideology.csv", index=False)

print("Task 1 shape:", df_t1.shape)
print("Task 2 shape:", df_t2.shape)
df_t1.head(3)


Task 1 shape: (101, 10)
Task 2 shape: (101, 5)


,senator_id,full_name,birthdate,birth_year_inferred,education,party,committee_roles,gender,race_ethnicity,error
0,Adam_B_Schiff_CA,Adam Schiff,None,NaN,"[{""degree"": null, ""institution"": ""Stanford Uni...",null,"[""House Committee on the Judiciary"", ""House Co...",None,None,None
1,Alan_Armstrong_OK,JANE DOE,None,NaN,[],None,"[""Energy"", ""Judiciary"", ""Banking""]",None,None,None
2,Alex_Padilla_CA,Alex Padilla,None,NaN,"[{""degree"": ""Bachelor of Science degree in Mec...",None,"[""Chairman of the Senate Judiciary Subcommitte...",None,Latino,None


## 7. Quick Descriptive Analysis

In [28]:
print("=== Task 1: Field Coverage ===")
for col in ["full_name","birthdate","birth_year_inferred","party","gender","race_ethnicity"]:
    filled = df_t1[col].notna().sum()
    print(f"  {col:30s}: {filled}/{len(df_t1)}")

print("\n=== Task 2: Ideology Distribution ===")
print(df_t2["ideological_label"].value_counts())

print("\n=== Task 2: Mean Confidence by Label ===")
df_t2["confidence"] = pd.to_numeric(df_t2["confidence"], errors="coerce")
print(df_t2.groupby("ideological_label")["confidence"].mean().round(2))

print("\n=== Errors ===")
for label, df in [("Task 1", df_t1), ("Task 2", df_t2)]:
    errs = df[df["error"].notna()][["senator_id","error"]]
    print(f"  {label}: {len(errs)} errors")
    if len(errs): print(errs.to_string(index=False))


=== Task 1: Field Coverage ===
  full_name                     : 93/101
  birthdate                     : 18/101
  birth_year_inferred           : 7/101
  party                         : 48/101
  gender                        : 6/101
  race_ethnicity                : 16/101

=== Task 2: Ideology Distribution ===
ideological_label
Conservative    45
Liberal         39
Unclear          8
Moderate         7
Name: count, dtype: int64

=== Task 2: Mean Confidence by Label ===
ideological_label
Conservative    0.83
Liberal         0.84
Moderate        0.80
Unclear         0.00
Name: confidence, dtype: float64

=== Errors ===
  Task 1: 0 errors
  Task 2: 2 errors
             senator_id                                                                                                                                                                                                                                          error
    Marsha_Blackburn_TN                                                 

## 8a. Evaluation Metrics (Liu et al. Section 6.1.4)

Liu et al. use three metrics:
- **Accuracy** — exact match for structured fields (email, phone). We apply this to `party` and `full_name`.
- **Rouge-1** — unigram F1 for text-overlap fields (work experience, education). We apply to `committee_roles` and `education`.
- **BERT score** — semantic similarity for all fields.

**Ground truth** must be loaded from `ground_truth.csv` — a CSV you create by
cross-referencing Ballotpedia/Wikipedia. Minimum required columns:
`senator_id, full_name, party, education_text, committee_roles_text`

The cell below will skip gracefully if the file does not yet exist.

In [ ]:
# ── Evaluation metrics (Liu et al. Section 6.1.4) ───────────────────────
# Requires: senate_results/ground_truth.csv
# Columns:  senator_id | full_name | party | education_text | committee_roles_text

GT_PATH = OUTPUT_DIR / "ground_truth.csv"

if not GT_PATH.exists():
    print("ground_truth.csv not found — skipping evaluation.")
    print("Create it by cross-referencing task1_pii.csv against Ballotpedia/Wikipedia.")
else:
    df_gt = pd.read_csv(GT_PATH)
    df_pred = pd.read_csv(OUTPUT_DIR / "task1_pii.csv")
    merged  = df_gt.merge(df_pred, on="senator_id", how="inner")
    print(f"Evaluating {len(merged)} senators with ground truth.")

    # ── Accuracy: exact-match fields (Liu et al. eq. 1) ──────────────────
    # Applied to full_name and party — the senator analogs of email/phone.
    for field in ["full_name", "party"]:
        gt_col   = field + ("_x" if field+"_x" in merged.columns else "")
        pred_col = field + ("_y" if field+"_y" in merged.columns else "")
        if gt_col in merged.columns and pred_col in merged.columns:
            acc = (merged[gt_col].str.lower().str.strip() ==
                   merged[pred_col].fillna("").str.lower().str.strip()).mean()
            print(f"Accuracy — {field}: {acc:.2%}")

    # ── Rouge-1: text-overlap fields (Liu et al. eq. 1) ──────────────────
    # Applied to committee_roles and education — analogous to work/education experience.
    scorer_r = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
    for field, gt_field in [
        ("committee_roles", "committee_roles_text"),
        ("education",       "education_text")
    ]:
        if gt_field not in merged.columns:
            continue
        pred_col = field + "_y" if field+"_y" in merged.columns else field
        scores = []
        for _, row in merged.iterrows():
            pred = str(row.get(pred_col, "") or "")
            gt   = str(row.get(gt_field, "") or "")
            if gt:
                s = scorer_r.score(gt, pred)["rouge1"].fmeasure
                scores.append(s)
        if scores:
            print(f"Rouge-1   — {field}: {sum(scores)/len(scores):.3f}")

    # ── BERT score: semantic similarity (Liu et al. eq. 2) ───────────────
    # Applied to committee_roles and education for semantic overlap.
    for field, gt_field in [
        ("committee_roles", "committee_roles_text"),
        ("education",       "education_text")
    ]:
        if gt_field not in merged.columns:
            continue
        pred_col = field + "_y" if field+"_y" in merged.columns else field
        preds = merged[pred_col].fillna("").astype(str).tolist()
        gts   = merged[gt_field].fillna("").astype(str).tolist()
        if any(gts):
            P, R, F = bert_score.score(preds, gts, lang="en", verbose=False)
            print(f"BERT score — {field}: F1={F.mean().item():.3f}")

    print("\nSave these numbers alongside field-coverage counts for your results table.")


## 8b. Baseline Comparison Table (Liu et al. Tables 4–5)

Aggregates LLM extraction vs regex and spaCy NER across all processed profiles.
Reproduce the paper's key finding: *LLM outperforms traditional methods in almost all scenarios.*

In [ ]:
# ── Run baselines across all processed profiles ──────────────────────────
# Reads the HTML files already processed by the main pipeline.
baseline_rows = []
for hf in tqdm(html_files, desc="Baselines"):
    text = extract_readable_text(hf.read_text(encoding="utf-8", errors="ignore"))
    regex_r = regex_extract(text)
    spacy_r = spacy_extract(text)
    baseline_rows.append({
        "senator_id":             hf.stem,
        "regex_name":             regex_r["full_name"],
        "regex_party":            regex_r["party"],
        "regex_email_found":      1 if regex_r["email"] else 0,
        "regex_phone_found":      1 if regex_r["phone"] else 0,
        "spacy_top_person":       spacy_r["persons_detected"][0] if spacy_r["persons_detected"] else None,
        "spacy_orgs_count":       len(spacy_r["orgs_detected"]),
    })

df_bl = pd.DataFrame(baseline_rows)
df_bl.to_csv(OUTPUT_DIR / "baselines.csv", index=False)

# ── Compare coverage: LLM vs baselines ───────────────────────────────────
df_t1  = pd.read_csv(OUTPUT_DIR / "task1_pii.csv")
merged_bl = df_t1.merge(df_bl, on="senator_id")

llm_name_cov   = merged_bl["full_name"].notna().mean()
regex_name_cov = merged_bl["regex_name"].notna().mean()
spacy_name_cov = merged_bl["spacy_top_person"].notna().mean()

llm_party_cov   = merged_bl["party"].notna().mean()
regex_party_cov = merged_bl["regex_party"].notna().mean()

print("=== Name extraction coverage (Liu et al. Table 4–5 analog) ===")
print(f"  LLM:   {llm_name_cov:.1%}")
print(f"  Regex: {regex_name_cov:.1%}")
print(f"  spaCy: {spacy_name_cov:.1%}")
print("\n=== Party extraction coverage ===")
print(f"  LLM:   {llm_party_cov:.1%}")
print(f"  Regex: {regex_party_cov:.1%}")
print("\nFull baseline results saved to baselines.csv")


## 9. Next Steps

1. **Ground truth:** Populate `senate_results/ground_truth.csv` from Ballotpedia/Wikipedia, then rerun Section 8a to compute Accuracy, Rouge-1, and BERT score.
2. **Prompt ablation:** Change `TASK1_PROMPT = TASK1_PSEUDOCODE` (or `TASK1_ICL`) and rerun the pipeline on a held-out subset. Compare field coverage to replicate Liu et al. Table 13.
3. **Model scaling:** Run Section 5b on your full dataset (not just 10 senators) once rate limits allow. Report the 8B vs 70B delta to replicate their larger-model finding (Table 3).
4. **Privacy framing:** Argue that public profile language leaks ideological identity with measurable reliability — a novel finding extending Liu et al. (2025) and adjacent to Staab et al. (2024, ICLR) on LLM inference attacks against latent attributes.
5. **Cite the paper correctly:** Liu, Jia, Jia, Gong. *Evaluating LLM-based Personal Information Extraction and Countermeasures.* USENIX Security 2025.